# 02 -- Baseline Contact Model

**Contact Luck Prototype v0.1**

Trains the Version 0.1 baseline multinomial logistic regression on the cleaned, training-eligible contact events, using the time-based development design (train on 2021-2023, validate on 2024).

> **2025 is a protected, untouched final-test season. Never tune, iterate, or select features using 2025 results.** This notebook never loads 2025 data.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [ ]:
import pandas as pd

from mlb_luck_score.config import PROCESSED_DATA_DIR, TRAIN_SEASONS, VALIDATION_SEASONS
from mlb_luck_score.models.train_contact_model import (
    evaluate_model,
    predict_proba_ordered,
    train_model,
    validate_probabilities,
)

CLEANED_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"
pd.set_option("display.width", 120)

In [ ]:
if CLEANED_PATH.exists():
    df = pd.read_parquet(CLEANED_PATH)
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    print(f"Loaded {len(df)} rows ({len(training_eligible)} training-eligible)")
else:
    df = None
    training_eligible = None
    print(
        f"No cleaned data found at {CLEANED_PATH}.\n"
        "Run `make download-sample` then `make clean-data`, then re-run this notebook."
    )

## Time-based split

Train seasons: configured in `mlb_luck_score.config.TRAIN_SEASONS`. Validation seasons: `mlb_luck_score.config.VALIDATION_SEASONS`. 2025 (`FINAL_TEST_SEASONS`) is never touched here.

In [ ]:
if training_eligible is not None:
    train_df = training_eligible[training_eligible["season"].isin(TRAIN_SEASONS)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
    print(f"Train seasons {TRAIN_SEASONS}: {len(train_df)} rows")
    print(f"Validation seasons {VALIDATION_SEASONS}: {len(val_df)} rows")
else:
    train_df = val_df = None
    print("Skipped -- no data loaded.")

## Train the baseline pipeline

In [ ]:
if train_df is not None and len(train_df) > 0:
    trained = train_model(train_df)
    print("Numeric features:", trained.numeric_features)
    print("Categorical features:", trained.categorical_features)
else:
    trained = None
    print(
        "Skipped -- no training-eligible rows for the configured train seasons. "
        "A one-week sample may not contain enough rows per season; this is expected "
        "for the bootstrap sample and is not a claim about model quality."
    )

## Probability predictions

In [ ]:
if trained is not None and val_df is not None and len(val_df) > 0:
    feature_cols = trained.numeric_features + trained.categorical_features
    proba_df = predict_proba_ordered(trained, val_df[feature_cols].head(10))
    validate_probabilities(proba_df)
    display(proba_df)
else:
    print("Skipped -- no trained model or no validation rows available.")

## Core evaluation metrics (validation season)

In [ ]:
if trained is not None and val_df is not None and len(val_df) > 0:
    metrics = evaluate_model(trained, val_df)
    print("Sample count:", metrics["sample_count"])
    print("Multiclass log loss:", metrics["multiclass_log_loss"])
    print("Brier score by class:", metrics["brier_score_by_class"])
    print("Outcome class frequencies:", metrics["outcome_class_frequencies"])
    print("Feature missingness:", metrics["feature_missingness"])
    print("\nNo claim of strong predictive performance is made here -- these are "
          "raw diagnostic numbers for a small bootstrap sample.")
else:
    print("Skipped -- no trained model or no validation rows available.")